In [ ]:
import string

text = "Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"

text_lower = text.lower()

translator = str.maketrans('', '', string.punctuation)
clean_text = text_lower.translate(translator)

print(text)
print(clean_text)

Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed
ugh the deliveries were delayed i seriously hate waiting annoyed


In [ ]:
tokens = clean_text.split()

print("Tokens:" , tokens)
print("Token Count: ",len(tokens))

Tokens: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Token Count:  10


In [ ]:
!pip install transformers -q

In [ ]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer, e.g., 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example text for subword tokenization
complex_text = "The microtransactional system was counterintuitive"

# Tokenize the text into subwords
subword_tokens = tokenizer.tokenize(complex_text)

print("Original Text:", complex_text)
print("Subword Tokens:", subword_tokens)
print("Number of Subword Tokens:", len(subword_tokens))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Original Text: The microtransactional system was counterintuitive
Subword Tokens: ['the', 'micro', '##tra', '##ns', '##act', '##ional', 'system', 'was', 'counter', '##int', '##uit', '##ive']
Number of Subword Tokens: 12


In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

filtered_tokens = [token for token in tokens if token not in stop_words]

print("Tokens: ", tokens)
print("Filtered Tokens: ", filtered_tokens)

Tokens:  ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Filtered Tokens:  ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download("wordnet")

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["deliveries", "walking", "studies", "delayed"]

for w in words:
  print(f"{w} | {stemmer.stem(w)} | {lemmatizer.lemmatize(w)}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


deliveries | deliveri | delivery
walking | walk | walking
studies | studi | study
delayed | delay | delayed


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

words = ["King", "Queen", "Apple"]
embeddings = model.encode(words)

for i in range(3):
  print("Word: ", words[i])
  print("Embedding: ", embeddings[i][:3])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Word:  King
Embedding:  [-0.0595993   0.0505124  -0.06951007]
Word:  Queen
Embedding:  [ 0.035487   -0.0656046  -0.00993492]
Word:  Apple
Embedding:  [-0.00613845  0.03101175  0.06479359]


In [ ]:
# To check similarity between the vector embeddings of words we have
# Cosine Similarity Function
from sklearn.metrics.pairwise import cosine_similarity

print(cosine_similarity([embeddings[0]], [embeddings[2]])) # King vs Apple
print(cosine_similarity([embeddings[0]], [embeddings[1]])) # King vs Queen

[[0.2431831]]
[[0.68071276]]


# 📝 NLP Text Preprocessing — Student Exercise Workbook

Welcome! You're a Junior AI Engineer at a tech startup tasked with building a **spam classifier**. But before any model can learn, you must transform raw, messy human language into clean, numerical input.

This notebook contains **7 exercises** that build up from simple cleaning all the way to a Keras-ready padded tensor.

## 📋 Structure
| Section | Topic | Points |
|---|---|---|
| Q1 | Noise Removal & Normalization | 10 |
| Q2 | Tokenization | 10 |
| Q3 | Stop Word Removal | 10 |
| Q4 | Stemming vs Lemmatization | 10 |
| Q5 | Bag-of-Words Vectorization | 15 |
| Q6 | Full Preprocessing Pipeline | 15 |
| Q7 | Keras Tokenizer & Padding | 20 |
| **Total** | | **90** |

## 📝 Instructions
1. Run the **Setup** cell first — it imports libraries (NLTK, Keras), downloads the stopwords/wordnet packs, and initializes the auto-grader.
2. Each question has a cell with your task described in **comments**. Fill in your code where indicated.
3. After completing each question, run the corresponding `grader.check_qN(...)` cell to check your work and earn points.
4. Your **Live Score** updates automatically after every check.
5. Submit the final completed notebook (with all outputs visible) back to your instructor.

**The reference sentence we'll track throughout this notebook:**
> `"Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"`

**Good luck! 🚀**

## 🔧 Setup — Run this cell FIRST

In [ ]:
# Core imports & auto-grader — run this cell FIRST
import re
import string
import numpy as np
from IPython.display import HTML, display, clear_output

# --- NLTK downloads (silent) ---
import nltk
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

from nltk.corpus import stopwords as _nltk_stopwords
from nltk.stem import PorterStemmer as _PorterStemmer, WordNetLemmatizer as _WordNetLemmatizer

# --- Keras imports for Q7 ---
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


class NLPGrader:
    def __init__(self):
        self.scores    = {'Q1': 0,  'Q2': 0,  'Q3': 0,  'Q4': 0,  'Q5': 0,  'Q6': 0,  'Q7': 0}
        self.max_scores= {'Q1': 10, 'Q2': 10, 'Q3': 10, 'Q4': 10, 'Q5': 15, 'Q6': 15, 'Q7': 20}
        self.display_score()

    def display_score(self):
        total = sum(self.scores.values())
        max_total = sum(self.max_scores.values())
        if total == max_total:
            bg, bd, tc = "#d4edda", "#28a745", "#155724"   # green when perfect
        elif total >= max_total * 0.7:
            bg, bd, tc = "#cce5ff", "#0d6efd", "#084298"   # blue when good
        else:
            bg, bd, tc = "#fff3cd", "#ffc107", "#856404"   # yellow otherwise
        breakdown = " | ".join([f"{k}: {v}/{self.max_scores[k]}" for k, v in self.scores.items()])
        html = f"""
        <div style="border: 2px solid {bd}; padding: 15px; border-radius: 8px; background-color: {bg}; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <h2 style="margin: 0; color: {tc};">📝 NLP Live Score: {total} / {max_total}</h2>
            <p style="margin: 5px 0 0 0; font-size: 14px; color: {tc};">{breakdown}</p>
            <p style="margin: 5px 0 0 0; font-size: 13px; color: {tc};">Run the grader.check_qN(...) cells to update your score.</p>
        </div>
        """
        display(HTML(html))

    # ---------- helpers ----------
    @staticmethod
    def _norm_ws(s):
        """Collapse multiple whitespace into single spaces and strip."""
        return ' '.join(str(s).split())

    # ---------- Q1: Noise Removal ----------
    def check_q1(self, clean_fn):
        clear_output(wait=True)
        try:
            if not callable(clean_fn):
                raise AssertionError("clean_text must be a function (def clean_text(text): ...)")
            cases = [
                ("Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed",
                 "ugh the deliveries were delayed i seriously hate waiting annoyed"),
                ("Hello!!! World???",                       "hello world"),
                ("Already clean text",                       "already clean text"),
                ("Mix3D c4se W!TH numb3rs",                  "mix3d c4se wth numb3rs"),
                ("FREE!!! Win $$$ now @user #spam",          "free win  now user spam"),  # $ and @ removed
            ]
            for i, (inp, exp) in enumerate(cases, 1):
                got_raw = clean_fn(inp)
                if not isinstance(got_raw, str):
                    raise AssertionError(f"Test {i}: clean_text must return a string, got {type(got_raw).__name__}.")
                got = self._norm_ws(got_raw)
                exp_norm = self._norm_ws(exp)
                if got != exp_norm:
                    raise AssertionError(f"Test {i} input={inp!r}\n   expected: {exp_norm!r}\n   got:      {got!r}")
            self.scores['Q1'] = self.max_scores['Q1']
            print(f"✅ Q1 Passed: All {len(cases)} text-cleaning cases handled correctly!")
        except AssertionError as e:
            self.scores['Q1'] = 0
            print(f"❌ Q1 Failed: {e}")
        except Exception as e:
            self.scores['Q1'] = 0
            print(f"❌ Q1 Failed with error: {e}")
        self.display_score()

    # ---------- Q2: Tokenization ----------
    def check_q2(self, tokenize_fn):
        clear_output(wait=True)
        try:
            if not callable(tokenize_fn):
                raise AssertionError("tokenize must be a function (def tokenize(text): ...)")
            cases = [
                ("ugh the deliveries were delayed i seriously hate waiting annoyed",
                 ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']),
                ("hello world",                  ['hello', 'world']),
                ("a b c d e",                    ['a', 'b', 'c', 'd', 'e']),
                ("single",                       ['single']),
                ("  extra   spaces  here  ",     ['extra', 'spaces', 'here']),
            ]
            for i, (inp, exp) in enumerate(cases, 1):
                got = tokenize_fn(inp)
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: tokenize must return a list, got {type(got).__name__}.")
                if got != exp:
                    raise AssertionError(f"Test {i} input={inp!r}\n   expected: {exp}\n   got:      {got}")
            self.scores['Q2'] = self.max_scores['Q2']
            print(f"✅ Q2 Passed: Tokenizer handled all {len(cases)} cases correctly!")
        except AssertionError as e:
            self.scores['Q2'] = 0
            print(f"❌ Q2 Failed: {e}")
        except Exception as e:
            self.scores['Q2'] = 0
            print(f"❌ Q2 Failed with error: {e}")
        self.display_score()

    # ---------- Q3: Stop Word Removal ----------
    def check_q3(self, remove_fn):
        clear_output(wait=True)
        try:
            if not callable(remove_fn):
                raise AssertionError("remove_stopwords must be a function (def remove_stopwords(tokens): ...)")
            sw = set(_nltk_stopwords.words('english'))
            cases = [
                ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed'],
                ['the', 'movie', 'was', 'good'],
                ['this', 'is', 'a', 'test', 'sentence'],
                ['python', 'machine', 'learning'],   # no stop words
                ['i', 'am', 'the', 'one'],           # mostly stop words
            ]
            for i, tokens in enumerate(cases, 1):
                got = remove_fn(list(tokens))   # pass a copy
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: must return a list, got {type(got).__name__}.")
                exp = [t for t in tokens if t not in sw]
                if got != exp:
                    raise AssertionError(f"Test {i} input={tokens}\n   expected: {exp}\n   got:      {got}")
            self.scores['Q3'] = self.max_scores['Q3']
            print(f"✅ Q3 Passed: All {len(cases)} stop-word cases cleared!")
        except AssertionError as e:
            self.scores['Q3'] = 0
            print(f"❌ Q3 Failed: {e}")
        except Exception as e:
            self.scores['Q3'] = 0
            print(f"❌ Q3 Failed with error: {e}")
        self.display_score()

    # ---------- Q4: Stemming + Lemmatization ----------
    def check_q4(self, result):
        clear_output(wait=True)
        try:
            if not isinstance(result, dict):
                raise AssertionError(f"root_forms must be a dict, got {type(result).__name__}.")
            words = ["deliveries", "waiting", "delayed", "studies", "running", "flies", "better"]
            stemmer = _PorterStemmer()
            lemmatizer = _WordNetLemmatizer()
            for w in words:
                if w not in result:
                    raise AssertionError(f"Missing key '{w}' in root_forms dict.")
                val = result[w]
                if not (isinstance(val, (tuple, list)) and len(val) == 2):
                    raise AssertionError(f"Value for '{w}' must be a (stem, lemma) tuple/list of length 2, got {val!r}.")
                exp_stem = stemmer.stem(w)
                exp_lemma = lemmatizer.lemmatize(w)   # default POS='n'
                got_stem, got_lemma = val[0], val[1]
                if got_stem != exp_stem:
                    raise AssertionError(f"'{w}': expected stem='{exp_stem}', got '{got_stem}'.")
                if got_lemma != exp_lemma:
                    raise AssertionError(f"'{w}': expected lemma='{exp_lemma}', got '{got_lemma}'.")
            self.scores['Q4'] = self.max_scores['Q4']
            print(f"✅ Q4 Passed: All {len(words)} words stemmed AND lemmatized correctly!")
        except AssertionError as e:
            self.scores['Q4'] = 0
            print(f"❌ Q4 Failed: {e}")
        except Exception as e:
            self.scores['Q4'] = 0
            print(f"❌ Q4 Failed with error: {e}")
        self.display_score()

    # ---------- Q5: Bag of Words ----------
    def check_q5(self, vectorize_fn):
        clear_output(wait=True)
        try:
            if not callable(vectorize_fn):
                raise AssertionError("vectorize must be a function (def vectorize(tokens, vocabulary): ...)")
            vocab = ["win", "free", "money", "ugh", "deliveries", "delayed", "seriously", "hate", "waiting", "annoyed"]

            cases = [
                # Worksheet sentence
                (['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed'],
                 [0, 0, 0, 1, 1, 1, 1, 1, 1, 1]),
                # Spam example
                (['win', 'free', 'money'],
                 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]),
                # Frequency counts (BoW = counts, not just 0/1!)
                (['hate', 'hate', 'hate', 'waiting'],
                 [0, 0, 0, 0, 0, 0, 0, 3, 1, 0]),
                # Word not in vocab — must be ignored, not raise
                (['ugh', 'pizza', 'pizza', 'free'],
                 [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]),
                # Empty tokens
                ([],
                 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
            ]
            for i, (toks, exp) in enumerate(cases, 1):
                got = vectorize_fn(list(toks), list(vocab))
                got_list = list(got) if hasattr(got, '__iter__') else got
                # cast numpy ints / etc. to plain ints for fair compare
                try:
                    got_list = [int(x) for x in got_list]
                except Exception:
                    raise AssertionError(f"Test {i}: vectorize must return a list/array of integers.")
                if len(got_list) != len(vocab):
                    raise AssertionError(f"Test {i}: output length must equal vocabulary length ({len(vocab)}), got {len(got_list)}.")
                if got_list != exp:
                    raise AssertionError(f"Test {i} tokens={toks}\n   expected: {exp}\n   got:      {got_list}\n   (Hint: BoW = COUNTS, not just presence; ignore tokens not in vocab.)")
            self.scores['Q5'] = self.max_scores['Q5']
            print(f"✅ Q5 Passed: Bag-of-Words vectorizer correct on all {len(cases)} cases (incl. counts & OOV handling)!")
        except AssertionError as e:
            self.scores['Q5'] = 0
            print(f"❌ Q5 Failed: {e}")
        except Exception as e:
            self.scores['Q5'] = 0
            print(f"❌ Q5 Failed with error: {e}")
        self.display_score()

    # ---------- Q6: Full Pipeline ----------
    def check_q6(self, preprocess_fn):
        clear_output(wait=True)
        try:
            if not callable(preprocess_fn):
                raise AssertionError("preprocess must be a function (def preprocess(text): ...)")
            sw = set(_nltk_stopwords.words('english'))
            lemmatizer = _WordNetLemmatizer()

            def reference(text):
                # 1. lowercase
                t = text.lower()
                # 2. remove punctuation
                t = t.translate(str.maketrans('', '', string.punctuation))
                # 3. tokenize (split on whitespace)
                toks = t.split()
                # 4. remove stop words
                toks = [w for w in toks if w not in sw]
                # 5. lemmatize (default POS='n')
                toks = [lemmatizer.lemmatize(w) for w in toks]
                return toks

            cases = [
                "Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed",
                "WIN free money!!! Click here NOW.",
                "The studies show that running is good for health.",
                "She studies every day, and her grades are improving!",
                "Hello, world! This is a SIMPLE test... right?",
            ]
            for i, text in enumerate(cases, 1):
                got = preprocess_fn(text)
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: preprocess must return a list, got {type(got).__name__}.")
                exp = reference(text)
                if got != exp:
                    raise AssertionError(f"Test {i} input={text!r}\n   expected: {exp}\n   got:      {got}\n   (Pipeline order: lowercase → strip punctuation → split → drop stop words → lemmatize.)")
            self.scores['Q6'] = self.max_scores['Q6']
            print(f"✅ Q6 Passed: Full pipeline matches the reference on all {len(cases)} sentences!")
        except AssertionError as e:
            self.scores['Q6'] = 0
            print(f"❌ Q6 Failed: {e}")
        except Exception as e:
            self.scores['Q6'] = 0
            print(f"❌ Q6 Failed with error: {e}")
        self.display_score()

    # ---------- Q7: Keras Tokenizer + Padding ----------
    def check_q7(self, tokenizer, padded_data, oov_test_seq):
        clear_output(wait=True)
        try:
            # 1) Tokenizer must be the right type
            if type(tokenizer).__name__ != 'Tokenizer':
                raise AssertionError(f"tokenizer must be a keras Tokenizer, got {type(tokenizer).__name__}.")

            # 2) num_words must be 50
            if getattr(tokenizer, 'num_words', None) != 50:
                raise AssertionError(f"Tokenizer num_words must be 50, got {getattr(tokenizer, 'num_words', None)}.")

            # 3) oov_token must be '<OOV>'
            if getattr(tokenizer, 'oov_token', None) != '<OOV>':
                raise AssertionError(f"Tokenizer oov_token must be '<OOV>', got {getattr(tokenizer, 'oov_token', None)!r}.")

            # 4) Word index must contain expected words (proves fit_on_texts was called)
            wi = tokenizer.word_index
            for must_have in ['<OOV>', 'free', 'money', 'meeting', 'deliveries']:
                if must_have not in wi:
                    raise AssertionError(f"word_index missing '{must_have}'. Did you call fit_on_texts on the training sentences?")

            # 5) <OOV> must have index 1 (Keras convention when oov_token is set)
            if wi.get('<OOV>') != 1:
                raise AssertionError(f"<OOV> token should be index 1, got {wi.get('<OOV>')}.")

            # 6) padded_data shape must be (n_train_sentences, 12)
            arr = np.asarray(padded_data)
            if arr.ndim != 2:
                raise AssertionError(f"padded_data must be 2D, got shape {arr.shape}.")
            if arr.shape[1] != 12:
                raise AssertionError(f"padded_data sequences must have length 12 (maxlen), got {arr.shape[1]}.")
            if arr.shape[0] < 5:
                raise AssertionError(f"padded_data should have one row per training sentence (>=5 rows), got {arr.shape[0]}.")

            # 7) Padding must be 'post' — i.e., trailing zeros, not leading.
            # First sentence "win free money now" → 4 tokens, then 8 zeros at the end.
            first_row = list(arr[0])
            if first_row[0] == 0:
                raise AssertionError("padded_data appears to use padding='pre' (leading zeros). Use padding='post'.")
            if first_row[-1] != 0:
                raise AssertionError("padded_data should have trailing zeros (padding='post').")

            # 8) OOV behaviour — oov_test_seq should encode unknown words as <OOV> = index 1
            seq = oov_test_seq
            # accept either a flat list (single sentence) or a list-of-lists
            if seq and isinstance(seq[0], list):
                seq = seq[0]
            seq = list(seq)
            if 1 not in seq:
                raise AssertionError("oov_test_seq has no <OOV> (=1) token. Did you encode a sentence containing UNKNOWN words using texts_to_sequences?")

            self.scores['Q7'] = self.max_scores['Q7']
            print("✅ Q7 Passed: Keras Tokenizer + Padding pipeline is correct (vocab, OOV, shape, post-padding).")
        except AssertionError as e:
            self.scores['Q7'] = 0
            print(f"❌ Q7 Failed: {e}")
        except Exception as e:
            self.scores['Q7'] = 0
            print(f"❌ Q7 Failed with error: {e}")
        self.display_score()


grader = NLPGrader()
print("Setup complete. NLTK + Keras ready. Good luck! 📝")


Setup complete. NLTK + Keras ready. Good luck! 📝


---
## 🧼 Q1 — Noise Removal & Normalization — 10 pts

Computers are extremely literal. To a computer `"Apple"`, `"apple"`, and `"apple!"` are three completely different tokens. Your first job is to **standardize** raw text by:
1. Lowercasing every character.
2. Removing every punctuation mark (use `string.punctuation`).

> 💡 Tip from the worksheet: `str.maketrans('', '', string.punctuation)` builds a translation table that maps every punctuation char to *Nothing*.


In [ ]:
# =========================================================================
# Q1: WRITE A clean_text(text) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  clean_text(text)  that:
#   1. Converts the input string to LOWERCASE.
#   2. Removes ALL punctuation using string.punctuation.
#   3. Returns the cleaned string.
#
# Examples:
#   clean_text("Hello!!! World???")
#       -> 'hello world'
#   clean_text("Ugh... The deliveries were DELAYED!! #annoyed")
#       -> 'ugh the deliveries were delayed annoyed'
#
# Notes:
#   - The grader normalizes whitespace, so multiple spaces left over from
#     removing punctuation will not be penalized.
#   - You MUST use string.punctuation (not a hand-typed list).
# =========================================================================

import string

# YOUR CODE HERE
def clean_text(text):
    # Step 1: lowercase
    text = text.lower()
    # Step 2: remove punctuation
    cleaned_text = text.translate(str.maketrans('','', string.punctuation))
    # Step 3: return
    return cleaned_text


# Quick sanity check
raw = "Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"
print("Original:", raw)
print("Cleaned :", clean_text(raw))

Original: Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed
Cleaned : ugh the deliveries were delayed i seriously hate waiting annoyed


In [ ]:
# Run this cell AFTER completing Q1 to check your answer
grader.check_q1(clean_text)


✅ Q1 Passed: All 5 text-cleaning cases handled correctly!


---
## ✂️ Q2 — Tokenization — 10 pts

Models don't read sentences; they read **tokens** (words). We need to chop the cleaned stream of text into a list of individual units.

We'll use the simple approach from the worksheet: **split on whitespace**.

> 💡 Be careful: `"  multiple   spaces  "` should still produce `['multiple', 'spaces']` — no empty strings, no leading/trailing blanks. (Hint: plain `.split()` with NO argument handles this for you.)


In [ ]:
# =========================================================================
# Q2: WRITE A tokenize(text) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  tokenize(text)  that:
#   - Takes a (presumably already-cleaned) string
#   - Returns a list of word tokens, splitting on whitespace.
#   - Must NOT contain any empty strings, even if input has multiple
#     consecutive spaces / leading / trailing whitespace.
#
# Examples:
#   tokenize("hello world")           -> ['hello', 'world']
#   tokenize("  extra   spaces  ")    -> ['extra', 'spaces']
# =========================================================================

# YOUR CODE HERE
def tokenize(text):
    tokenized_text = text.split()
    return tokenized_text


# Quick sanity check using your Q1 function
cleaned = clean_text("Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed")
tokens  = tokenize(cleaned)
print("Tokens     :", tokens)
print("Token count:", len(tokens))


Tokens     : ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
Token count: 10


In [ ]:
# Run this cell AFTER completing Q2 to check your answer
grader.check_q2(tokenize)


✅ Q2 Passed: Tokenizer handled all 5 cases correctly!


---
## 🛑 Q3 — Stop Word Removal — 10 pts

Words split into two camps:
- **Content words** carry meaning: *pizza, eat, tasty.*
- **Stop words** are grammatical glue with little standalone meaning: *the, is, at.*

For simple/light models like Naive Bayes we usually **drop** stop words to save space.

Use NLTK's English stop-word list. The setup cell already downloaded it for you.

> ⚠️ **Food for thought:** *"The movie was NOT good."* — naïvely removing stop words turns this into `['movie', 'good']` and the meaning **flips**. We accept this trade-off here for the simple model — but remember it for real-world systems.


In [ ]:
# =========================================================================
# Q3: WRITE A remove_stopwords(tokens) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  remove_stopwords(tokens)  that:
#   - Takes a list of lower-case tokens.
#   - Returns a new list with all NLTK English stop words removed.
#   - PRESERVES the original order of the remaining tokens.
#
# Required imports / setup:
#   from nltk.corpus import stopwords
#   stop_words = set(stopwords.words('english'))
#
# Example:
#   remove_stopwords(['ugh','the','deliveries','were','delayed','i','hate','waiting'])
#       -> ['ugh', 'deliveries', 'delayed', 'hate', 'waiting']
# =========================================================================

from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# YOUR CODE HERE
def remove_stopwords(tokens):
    filtered_tokens = [token for token in tokens if token not in stop_words]
    return filtered_tokens


# Quick sanity check chained from Q1 + Q2
demo_tokens = tokenize(clean_text("Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"))
print("Before:", demo_tokens)
print("After :", remove_stopwords(demo_tokens))


Before: ['ugh', 'the', 'deliveries', 'were', 'delayed', 'i', 'seriously', 'hate', 'waiting', 'annoyed']
After : ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# Run this cell AFTER completing Q3 to check your answer
grader.check_q3(remove_stopwords)


✅ Q3 Passed: All 5 stop-word cases cleared!


---
## 🌱 Q4 — Stemming vs. Lemmatization — 10 pts

If a model is trained on `"waiting"` but a user types `"wait"`, the model won't realise they're related. We cut every word down to its **root form**.

| Approach | What it does | Speed | Accuracy |
|---|---|---|---|
| **Stemming** | Crude chopper — slices suffixes; sometimes makes non-words (`studies → studi`) | ⚡ fast | meh |
| **Lemmatization** | Dictionary lookup — returns real words (`studies → study`) | 🐢 slower | ✅ accurate |

Build a dictionary that maps each word to **both** its stem and its lemma.


In [ ]:
# =========================================================================
# Q4: BUILD root_forms — A DICT OF (stem, lemma) FOR EACH WORD
# =========================================================================
#
# Task:
#   For every word in the list `words` below, compute BOTH:
#       - its Porter stem        (PorterStemmer().stem(w))
#       - its WordNet lemma      (WordNetLemmatizer().lemmatize(w))   # default POS = 'n'
#   and store the result in a dictionary named EXACTLY:  root_forms
#
#   Format:
#       root_forms = {
#           'word_1': ('stem_1', 'lemma_1'),
#           'word_2': ('stem_2', 'lemma_2'),
#           ...
#       }
#
# Required words: deliveries, waiting, delayed, studies, running, flies, better
# =========================================================================

from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download("wordnet")

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["deliveries", "waiting", "delayed", "studies", "running", "flies", "better"]

# YOUR CODE HERE
root_forms = {word: (stemmer.stem(word), lemmatizer.lemmatize(word)) for word in words}


# Pretty print
print(f"{'WORD':<12} {'STEM':<12} {'LEMMA':<12}")
print("-" * 36)
for w in words:
    s, l = root_forms[w]
    print(f"{w:<12} {s:<12} {l:<12}")


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


WORD         STEM         LEMMA       
------------------------------------
deliveries   deliveri     delivery    
waiting      wait         waiting     
delayed      delay        delayed     
studies      studi        study       
running      run          running     
flies        fli          fly         
better       better       better      


In [ ]:
# Run this cell AFTER completing Q4 to check your answer
grader.check_q4(root_forms)


✅ Q4 Passed: All 7 words stemmed AND lemmatized correctly!


---
## 📦 Q5 — Bag-of-Words Vectorization — 15 pts

Math equations and neural networks cannot multiply strings. We must convert tokens into **numbers**.

**Naïve idea:** *"Just number the words! `hate=1`, `waiting=2`, `annoyed=3`..."* — ❌ this fails because it implies `annoyed = 3 × hate`, which is meaningless.

**Better idea: Bag-of-Words (BoW)**. We assume sentences containing the same set of words have similar meaning.
1. Fix a **global vocabulary** (the universe of words our AI knows).
2. For each sentence, output a vector with one slot per vocab word, containing the **count** of that word in the sentence.

> ⚠️ **Two subtleties to handle:**
> - BoW = **counts**, not just 0/1 presence. (`['hate','hate','hate']` → 3 in the `hate` slot.)
> - Tokens not in the vocabulary must be **silently ignored** — never raise an error.


In [ ]:
# =========================================================================
# Q5: WRITE A vectorize(tokens, vocabulary) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  vectorize(tokens, vocabulary)  that:
#   - Returns a list of length len(vocabulary).
#   - vector[i] = how many times vocabulary[i] appears in tokens.
#   - Tokens NOT in vocabulary must be silently ignored.
#   - An empty token list must produce a vector of all zeros.
#
# Reference vocabulary (from worksheet Part 5):
vocabulary = ["win", "free", "money", "ugh", "deliveries",
              "delayed", "seriously", "hate", "waiting", "annoyed"]
#
# Examples:
#   vectorize(['win','free','money'], vocabulary)
#       -> [1, 1, 1, 0, 0, 0, 0, 0, 0, 0]
#   vectorize(['hate','hate','hate','waiting'], vocabulary)
#       -> [0, 0, 0, 0, 0, 0, 0, 3, 1, 0]
#   vectorize(['ugh','pizza','pizza','free'], vocabulary)   # 'pizza' not in vocab
#       -> [0, 1, 0, 1, 0, 0, 0, 0, 0, 0]
# =========================================================================

# YOUR CODE HERE
def vectorize(tokens, vocabulary):
    # Initialize a list of zeros with the same length as vocabulary
    vector = [0] * len(vocabulary)

    # Create a dictionary to map vocabulary words to their indices for O(1) lookup
    vocab_index = {word: idx for idx, word in enumerate(vocabulary)}

    # Count occurrences of each token
    for token in tokens:
        if token in vocab_index:  # Only count if token is in vocabulary
            vector[vocab_index[token]] += 1

    return vector


# Sanity check — vectorize the worksheet sentence
demo_tokens = ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']
print("Vocab :", vocabulary)
print("Tokens:", demo_tokens)
print("BoW   :", vectorize(demo_tokens, vocabulary))


Vocab : ['win', 'free', 'money', 'ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']
Tokens: ['ugh', 'deliveries', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']
BoW   : [0, 0, 0, 1, 1, 1, 1, 1, 1, 1]


In [ ]:
# Run this cell AFTER completing Q5 to check your answer
grader.check_q5(vectorize)


✅ Q5 Passed: Bag-of-Words vectorizer correct on all 5 cases (incl. counts & OOV handling)!


---
## 🔗 Q6 — Full Preprocessing Pipeline — 15 pts

Time to wire everything together. Build **one** function that takes raw human text and returns the final clean list of tokens, ready for vectorization.

**Required pipeline order — exact same order as the worksheet:**
1. Lowercase
2. Strip punctuation (`string.punctuation`)
3. Tokenize on whitespace
4. Drop NLTK English stop words
5. Lemmatize each remaining token (default `WordNetLemmatizer().lemmatize(w)` — POS defaults to noun)

> 💡 You may call your earlier functions (`clean_text`, `tokenize`, `remove_stopwords`) — that's the whole point of writing them as functions! 🎯


In [ ]:
# =========================================================================
# Q6: WRITE THE COMPLETE preprocess(text) PIPELINE
# =========================================================================
#
# Task: Define a function named EXACTLY  preprocess(text)  that:
#   - Performs ALL FIVE STEPS in this order:
#       (1) lowercase
#       (2) remove punctuation                  (string.punctuation)
#       (3) split on whitespace                 (no empty strings)
#       (4) drop NLTK English stop words
#       (5) lemmatize each remaining token      (WordNetLemmatizer, default POS)
#   - Returns a list of clean lemma tokens.
#
# Example:
#   preprocess("Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed")
#       -> ['ugh', 'delivery', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']
#
# Note: 'deliveries' becomes 'delivery' (lemmatizer found the noun lemma).
#       'waiting' / 'delayed' stay as-is because default POS='n'.
# =========================================================================

import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words_set = set(stopwords.words('english'))
lemmatizer_q6  = WordNetLemmatizer()

# YOUR CODE HERE
def preprocess(text):
    text = text.lower()
    cleaned_text = text.translate(str.maketrans('','', string.punctuation))
    tokenized_text = cleaned_text.split()
    filtered_tokens = [token for token in tokenized_text if token not in stop_words_set]
    lemmatized_tokens = [lemmatizer_q6.lemmatize(token) for token in filtered_tokens]
    return lemmatized_tokens


# Sanity check
raw = "Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed"
print("Raw  :", raw)
print("Final:", preprocess(raw))


Raw  : Ugh... The deliveries were DELAYED!! I seriously hate waiting... #annoyed
Final: ['ugh', 'delivery', 'delayed', 'seriously', 'hate', 'waiting', 'annoyed']


In [ ]:
# Run this cell AFTER completing Q6 to check your answer
grader.check_q6(preprocess)


✅ Q6 Passed: Full pipeline matches the reference on all 5 sentences!


---
## 🚀 Q7 — Keras Tokenizer & Padding — 20 pts (The Final Boss)

Real-world neural networks expect **integer sequences of equal length** as input — not lists of strings of varying length. Keras gives us two helpers for that:

1. **`Tokenizer`** — assigns an integer to every word in the training corpus. Reserves a special `<OOV>` token for unknown words encountered later (Out-Of-Vocabulary).
2. **`pad_sequences`** — makes every sequence the same length by either truncating long ones or padding short ones with zeros.

Both are imported in the setup cell:
```python
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
```

Your job: take 7 short messages, fit a tokenizer, convert them to integer sequences, pad them all to length **12**, and finally test what happens when an *unknown* word shows up at inference time.


In [ ]:
# =========================================================================
# Q7: KERAS TOKENIZER + PADDING — Full pipeline (the hardest one!)
# =========================================================================
#
# Training data — 7 short messages. Some are spam-ish, some normal.
train_sentences = [
    "win free money now",
    "free vacation deal click here",
    "the meeting is tomorrow at noon",
    "call me back when you can",
    "the deliveries were delayed again",
    "win a free phone today",
    "i hate waiting for late deliveries",
]
#
# Sentence we'll later test (contains UNKNOWN words like 'pizza' and 'urgent'):
test_sentence  = "urgent free pizza offer"
#
# ----------------------- TASKS -----------------------
#  1. Create a Keras Tokenizer named EXACTLY  tokenizer  with:
#         num_words = 50
#         oov_token = '<OOV>'
#
#  2. FIT the tokenizer on train_sentences  (fit_on_texts).
#
#  3. Convert train_sentences into integer sequences using
#     texts_to_sequences. Store the result in a variable named  sequences.
#
#  4. Pad sequences with:
#         maxlen   = 12
#         padding  = 'post'        (zeros at the END, not the beginning)
#     Store the result in a variable named EXACTLY  padded_data.
#
#  5. Encode the test_sentence using texts_to_sequences and store the
#     resulting sequence in a variable named EXACTLY  oov_test_seq.
#     (You'll see the unknown words 'urgent' and 'pizza' get mapped to
#      index 1 — the <OOV> token. That is exactly what we want.)
#
# Expected shapes / behaviour:
#   - padded_data.shape == (7, 12)
#   - padded_data[0]    starts with non-zero ids and ENDS with zeros
#   - oov_test_seq      contains the integer 1 (<OOV>) somewhere
# =========================================================================

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Step 1: Create the tokenizer ---
# YOUR CODE HERE
tokenizer = Tokenizer(num_words=50, oov_token='<OOV>')


# --- Step 2: Fit on training sentences ---
tokenizer.fit_on_texts(train_sentences)


# --- Step 3: Convert to integer sequences ---
sequences = tokenizer.texts_to_sequences(train_sentences)


# --- Step 4: Pad sequences to length 12, padding='post' ---
padded_data = pad_sequences(sequences, maxlen=12, padding='post')


# --- Step 5: Encode the OOV test sentence ---
oov_test_seq = tokenizer.texts_to_sequences([test_sentence])



# Inspect your work
print("Word index (first 15):")
for w, i in list(tokenizer.word_index.items())[:15]:
    print(f"  {w!r:>15} -> {i}")
print()
print("Padded data shape:", padded_data.shape)
print("First padded row :", padded_data[0])
print("OOV test sentence:", test_sentence)
print("OOV test sequence:", oov_test_seq)


Word index (first 15):
          '<OOV>' -> 1
           'free' -> 2
            'win' -> 3
            'the' -> 4
     'deliveries' -> 5
          'money' -> 6
            'now' -> 7
       'vacation' -> 8
           'deal' -> 9
          'click' -> 10
           'here' -> 11
        'meeting' -> 12
             'is' -> 13
       'tomorrow' -> 14
             'at' -> 15

Padded data shape: (7, 12)
First padded row : [3 2 6 7 0 0 0 0 0 0 0 0]
OOV test sentence: urgent free pizza offer
OOV test sequence: [[1, 2, 1, 1]]


In [ ]:
# Run this cell AFTER completing Q7 to check your answer
grader.check_q7(tokenizer, padded_data, oov_test_seq)

✅ Q7 Passed: Keras Tokenizer + Padding pipeline is correct (vocab, OOV, shape, post-padding).


---
## ✅ Submission

When all seven checks show green:
1. Confirm the **NLP Live Score** at the top reads **90 / 90**.
2. From the Colab menu choose **File → Download → Download .ipynb**.
3. Send the downloaded notebook back to your instructor.

Make sure all cells have been **run** so the outputs (and your score) are visible in the saved file. 🎉

### 🎯 What you just learned
You took a raw, messy SMS-style text from chaos all the way down to a clean **(7, 12) integer tensor** ready to plug into any embedding layer / RNN / Transformer.
Every modern NLP system — including ChatGPT — starts with exactly this kind of pipeline. 🚀

# 🎬 NLP Text Preprocessing — Movie Review Sentiment Workbook

Welcome! You're a Junior NLP Engineer at a movie-streaming startup. The product team wants a system that automatically classifies user reviews as **positive** or **negative**. But before any model can learn, you need to transform raw, messy human reviews into clean, numerical input.

This notebook contains **7 exercises** that build up from simple cleaning all the way to a Keras-ready integer tensor — with truncation, padding, and a working decoder at the end.

## 📋 Structure
| Section | Topic | Points | Difficulty |
|---|---|---|---|
| Q1 | Noise Removal & Normalization | 10 | ⭐ |
| Q2 | Tokenization | 10 | ⭐ |
| Q3 | Stop Word Removal | 10 | ⭐⭐ |
| Q4 | Stemming vs Lemmatization | 10 | ⭐⭐ |
| Q5 | Vocabulary Builder + Corpus Vectorizer | 15 | ⭐⭐⭐ |
| Q6 | Full Single-Text Pipeline | 15 | ⭐⭐⭐ |
| Q7 | Keras Tokenizer + Padding + Truncation + Decoder | 30 | ⭐⭐⭐⭐⭐ |
| **Total** | | **100** | |

## 📝 Instructions
1. Run the **Setup** cell first — it imports libraries (NLTK, Keras), downloads the stopwords/wordnet packs, and initializes the auto-grader.
2. Each question has a cell with your task described in **comments**. Fill in your code where indicated.
3. After completing each question, run the corresponding `grader.check_qN(...)` cell to check your work and earn points.
4. Your **Live Score** updates automatically after every check.
5. Submit the final completed notebook (with all outputs visible) back to your instructor.

**The reference review we'll track throughout this notebook:**
> `"OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director"`

**Good luck, and may the box office be with you! 🎥🍿**

## 🔧 Setup — Run this cell FIRST

In [ ]:
# Core imports & auto-grader — run this cell FIRST
import re
import string
import numpy as np
from IPython.display import HTML, display, clear_output

# --- NLTK downloads (silent) ---
import nltk
for pkg in ['stopwords', 'wordnet', 'omw-1.4']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

from nltk.corpus import stopwords as _nltk_stopwords
from nltk.stem import PorterStemmer as _PorterStemmer, WordNetLemmatizer as _WordNetLemmatizer

# --- Keras imports for Q7 ---
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


class NLPGrader:
    def __init__(self):
        self.scores    = {'Q1': 0,  'Q2': 0,  'Q3': 0,  'Q4': 0,  'Q5': 0,  'Q6': 0,  'Q7': 0}
        self.max_scores= {'Q1': 10, 'Q2': 10, 'Q3': 10, 'Q4': 10, 'Q5': 15, 'Q6': 15, 'Q7': 30}
        self.display_score()

    def display_score(self):
        total = sum(self.scores.values())
        max_total = sum(self.max_scores.values())
        if total == max_total:
            bg, bd, tc = "#d4edda", "#28a745", "#155724"
        elif total >= max_total * 0.7:
            bg, bd, tc = "#cce5ff", "#0d6efd", "#084298"
        else:
            bg, bd, tc = "#fff3cd", "#ffc107", "#856404"
        breakdown = " | ".join([f"{k}: {v}/{self.max_scores[k]}" for k, v in self.scores.items()])
        html = f"""
        <div style="border: 2px solid {bd}; padding: 15px; border-radius: 8px; background-color: {bg}; font-family: sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.1);">
            <h2 style="margin: 0; color: {tc};">🎬 NLP Live Score: {total} / {max_total}</h2>
            <p style="margin: 5px 0 0 0; font-size: 14px; color: {tc};">{breakdown}</p>
            <p style="margin: 5px 0 0 0; font-size: 13px; color: {tc};">Run the grader.check_qN(...) cells to update your score.</p>
        </div>
        """
        display(HTML(html))

    @staticmethod
    def _norm_ws(s):
        return ' '.join(str(s).split())

    # ---------- Q1: Noise Removal ----------
    def check_q1(self, clean_fn):
        clear_output(wait=True)
        try:
            if not callable(clean_fn):
                raise AssertionError("clean_text must be a function (def clean_text(text): ...)")
            cases = [
                ("OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director",
                 "omg the performances were absolutely breathtaking i literally cried watching it  masterpiece director"),
                ("AMAZING film!!! Loved it so much...",      "amazing film loved it so much"),
                ("WORST movie ever ever ever",                "worst movie ever ever ever"),
                ("Mix3D rev1ew W!TH numb3rs",                 "mix3d rev1ew wth numb3rs"),
                ("BEST film *EVER* @critics #oscars2024!!!",  "best film ever critics oscars2024"),
            ]
            for i, (inp, exp) in enumerate(cases, 1):
                got_raw = clean_fn(inp)
                if not isinstance(got_raw, str):
                    raise AssertionError(f"Test {i}: clean_text must return a string, got {type(got_raw).__name__}.")
                got = self._norm_ws(got_raw)
                exp_norm = self._norm_ws(exp)
                if got != exp_norm:
                    raise AssertionError(f"Test {i} input={inp!r}\n   expected: {exp_norm!r}\n   got:      {got!r}")
            self.scores['Q1'] = self.max_scores['Q1']
            print(f"✅ Q1 Passed: All {len(cases)} text-cleaning cases handled correctly!")
        except AssertionError as e:
            self.scores['Q1'] = 0
            print(f"❌ Q1 Failed: {e}")
        except Exception as e:
            self.scores['Q1'] = 0
            print(f"❌ Q1 Failed with error: {e}")
        self.display_score()

    # ---------- Q2: Tokenization ----------
    def check_q2(self, tokenize_fn):
        clear_output(wait=True)
        try:
            if not callable(tokenize_fn):
                raise AssertionError("tokenize must be a function (def tokenize(text): ...)")
            cases = [
                ("omg the performances were absolutely breathtaking i literally cried watching it masterpiece director",
                 ['omg', 'the', 'performances', 'were', 'absolutely', 'breathtaking', 'i', 'literally', 'cried', 'watching', 'it', 'masterpiece', 'director']),
                ("amazing film",                 ['amazing', 'film']),
                ("a b c d e",                    ['a', 'b', 'c', 'd', 'e']),
                ("masterpiece",                  ['masterpiece']),
                ("  extra   spaces  here  ",     ['extra', 'spaces', 'here']),
            ]
            for i, (inp, exp) in enumerate(cases, 1):
                got = tokenize_fn(inp)
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: tokenize must return a list, got {type(got).__name__}.")
                if got != exp:
                    raise AssertionError(f"Test {i} input={inp!r}\n   expected: {exp}\n   got:      {got}")
            self.scores['Q2'] = self.max_scores['Q2']
            print(f"✅ Q2 Passed: Tokenizer handled all {len(cases)} cases correctly!")
        except AssertionError as e:
            self.scores['Q2'] = 0
            print(f"❌ Q2 Failed: {e}")
        except Exception as e:
            self.scores['Q2'] = 0
            print(f"❌ Q2 Failed with error: {e}")
        self.display_score()

    # ---------- Q3: Stop Word Removal ----------
    def check_q3(self, remove_fn):
        clear_output(wait=True)
        try:
            if not callable(remove_fn):
                raise AssertionError("remove_stopwords must be a function (def remove_stopwords(tokens): ...)")
            sw = set(_nltk_stopwords.words('english'))
            cases = [
                ['omg', 'the', 'performances', 'were', 'absolutely', 'breathtaking', 'i', 'literally', 'cried', 'watching', 'it', 'masterpiece', 'director'],
                ['the', 'movie', 'was', 'absolutely', 'amazing'],
                ['this', 'is', 'a', 'great', 'film'],
                ['cinematography', 'soundtrack', 'plot'],
                ['i', 'am', 'the', 'one', 'who', 'cried'],
            ]
            for i, tokens in enumerate(cases, 1):
                got = remove_fn(list(tokens))
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: must return a list, got {type(got).__name__}.")
                exp = [t for t in tokens if t not in sw]
                if got != exp:
                    raise AssertionError(f"Test {i} input={tokens}\n   expected: {exp}\n   got:      {got}")
            self.scores['Q3'] = self.max_scores['Q3']
            print(f"✅ Q3 Passed: All {len(cases)} stop-word cases cleared!")
        except AssertionError as e:
            self.scores['Q3'] = 0
            print(f"❌ Q3 Failed: {e}")
        except Exception as e:
            self.scores['Q3'] = 0
            print(f"❌ Q3 Failed with error: {e}")
        self.display_score()

    # ---------- Q4: Stemming + Lemmatization ----------
    def check_q4(self, result):
        clear_output(wait=True)
        try:
            if not isinstance(result, dict):
                raise AssertionError(f"root_forms must be a dict, got {type(result).__name__}.")
            words = ["performances", "actors", "watching", "directed", "movies", "stories", "running"]
            stemmer = _PorterStemmer()
            lemmatizer = _WordNetLemmatizer()
            for w in words:
                if w not in result:
                    raise AssertionError(f"Missing key '{w}' in root_forms dict.")
                val = result[w]
                if not (isinstance(val, (tuple, list)) and len(val) == 2):
                    raise AssertionError(f"Value for '{w}' must be a (stem, lemma) tuple/list of length 2, got {val!r}.")
                exp_stem = stemmer.stem(w)
                exp_lemma = lemmatizer.lemmatize(w)
                got_stem, got_lemma = val[0], val[1]
                if got_stem != exp_stem:
                    raise AssertionError(f"'{w}': expected stem='{exp_stem}', got '{got_stem}'.")
                if got_lemma != exp_lemma:
                    raise AssertionError(f"'{w}': expected lemma='{exp_lemma}', got '{got_lemma}'.")
            self.scores['Q4'] = self.max_scores['Q4']
            print(f"✅ Q4 Passed: All {len(words)} words stemmed AND lemmatized correctly!")
        except AssertionError as e:
            self.scores['Q4'] = 0
            print(f"❌ Q4 Failed: {e}")
        except Exception as e:
            self.scores['Q4'] = 0
            print(f"❌ Q4 Failed with error: {e}")
        self.display_score()

    # ---------- Q5: Build vocabulary + Vectorize corpus ----------
    def check_q5(self, build_vocab_fn, vectorize_corpus_fn):
        clear_output(wait=True)
        try:
            if not callable(build_vocab_fn):
                raise AssertionError("build_vocabulary must be a function.")
            if not callable(vectorize_corpus_fn):
                raise AssertionError("vectorize_corpus must be a function.")

            # ------- Test 1: Vocabulary build -------
            corpus = [
                ['great', 'movie', 'great', 'acting'],
                ['boring', 'movie', 'bad', 'acting'],
                ['great', 'acting', 'amazing', 'film'],
            ]
            vocab = build_vocab_fn(corpus)
            if not isinstance(vocab, list):
                raise AssertionError(f"build_vocabulary must return a list, got {type(vocab).__name__}.")
            expected_vocab = ['acting', 'amazing', 'bad', 'boring', 'film', 'great', 'movie']
            if vocab != expected_vocab:
                raise AssertionError(
                    f"build_vocabulary output incorrect.\n"
                    f"   expected (sorted unique): {expected_vocab}\n"
                    f"   got:                      {vocab}\n"
                    f"   (Hint: collect all unique tokens across the corpus and SORT alphabetically.)"
                )

            # ------- Test 2: vectorize_corpus on the same corpus -------
            matrix = vectorize_corpus_fn(corpus, vocab)
            arr = np.asarray(matrix)
            expected_matrix = np.array([
                [1, 0, 0, 0, 0, 2, 1],   # 'great' x2, 'movie' x1, 'acting' x1
                [1, 0, 1, 1, 0, 0, 1],   # boring/movie/bad/acting x1 each
                [1, 1, 0, 0, 1, 1, 0],   # great/acting/amazing/film x1 each
            ])
            if arr.shape != expected_matrix.shape:
                raise AssertionError(f"vectorize_corpus shape: expected {expected_matrix.shape}, got {arr.shape}.")
            if not np.array_equal(arr, expected_matrix):
                raise AssertionError(
                    f"vectorize_corpus matrix incorrect.\n"
                    f"   expected:\n{expected_matrix}\n"
                    f"   got:\n{arr}\n"
                    f"   (Hint: row i, col j = count of vocabulary[j] inside corpus[i].)"
                )

            # ------- Test 3: empty doc must produce zero row -------
            corpus2 = [
                ['great', 'movie'],
                [],                       # empty doc!
                ['amazing', 'amazing', 'amazing'],
            ]
            vocab2 = build_vocab_fn(corpus2)
            expected_vocab2 = ['amazing', 'great', 'movie']
            if vocab2 != expected_vocab2:
                raise AssertionError(f"Test 3 vocab: expected {expected_vocab2}, got {vocab2}.")
            mat2 = np.asarray(vectorize_corpus_fn(corpus2, vocab2))
            expected2 = np.array([[0, 1, 1], [0, 0, 0], [3, 0, 0]])
            if not np.array_equal(mat2, expected2):
                raise AssertionError(f"Test 3 matrix: expected\n{expected2}\ngot\n{mat2}")

            # ------- Test 4: out-of-vocab tokens must be ignored -------
            corpus3 = [['movie', 'pizza', 'movie']]   # 'pizza' not in vocab
            mat3 = np.asarray(vectorize_corpus_fn(corpus3, vocab))   # use vocab from Test 1
            # vocab = ['acting','amazing','bad','boring','film','great','movie']
            #          0         1         2      3         4       5        6
            expected3 = np.array([[0, 0, 0, 0, 0, 0, 2]])
            if not np.array_equal(mat3, expected3):
                raise AssertionError(f"Test 4 (OOV ignore): expected {expected3.tolist()}, got {mat3.tolist()}")

            self.scores['Q5'] = self.max_scores['Q5']
            print("✅ Q5 Passed: Vocabulary builder + corpus vectorizer correct (incl. empty docs & OOV)!")
        except AssertionError as e:
            self.scores['Q5'] = 0
            print(f"❌ Q5 Failed: {e}")
        except Exception as e:
            self.scores['Q5'] = 0
            print(f"❌ Q5 Failed with error: {e}")
        self.display_score()

    # ---------- Q6: Full Pipeline (single text) ----------
    def check_q6(self, preprocess_fn):
        clear_output(wait=True)
        try:
            if not callable(preprocess_fn):
                raise AssertionError("preprocess must be a function.")
            sw = set(_nltk_stopwords.words('english'))
            lemmatizer = _WordNetLemmatizer()

            def reference(text):
                t = text.lower()
                t = t.translate(str.maketrans('', '', string.punctuation))
                toks = t.split()
                toks = [w for w in toks if w not in sw]
                toks = [lemmatizer.lemmatize(w) for w in toks]
                return toks

            cases = [
                "OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director",
                "WORST movie EVER!!! Boring plot, terrible actors.",
                "The cinematography was stunning and the soundtrack moved me to tears.",
                "She directed three movies last year — all underrated stories!",
                "Hello, fans! This is THE most anticipated film of 2024... right?",
            ]
            for i, text in enumerate(cases, 1):
                got = preprocess_fn(text)
                if not isinstance(got, list):
                    raise AssertionError(f"Test {i}: preprocess must return a list, got {type(got).__name__}.")
                exp = reference(text)
                if got != exp:
                    raise AssertionError(f"Test {i} input={text!r}\n   expected: {exp}\n   got:      {got}\n   (Pipeline order: lowercase → strip punctuation → split → drop stop words → lemmatize.)")
            self.scores['Q6'] = self.max_scores['Q6']
            print(f"✅ Q6 Passed: Full pipeline matches the reference on all {len(cases)} reviews!")
        except AssertionError as e:
            self.scores['Q6'] = 0
            print(f"❌ Q6 Failed: {e}")
        except Exception as e:
            self.scores['Q6'] = 0
            print(f"❌ Q6 Failed with error: {e}")
        self.display_score()

    # ---------- Q7: Keras Tokenizer + Padding + Truncation + Decode ----------
    def check_q7(self, tokenizer, padded_data, oov_test_seq, decode_fn, train_sentences):
        clear_output(wait=True)
        try:
            # ---- 1. Tokenizer config ----
            if type(tokenizer).__name__ != 'Tokenizer':
                raise AssertionError(f"tokenizer must be a keras Tokenizer, got {type(tokenizer).__name__}.")
            if getattr(tokenizer, 'num_words', None) != 100:
                raise AssertionError(f"Tokenizer num_words must be 100, got {getattr(tokenizer, 'num_words', None)}.")
            if getattr(tokenizer, 'oov_token', None) != '<OOV>':
                raise AssertionError(f"Tokenizer oov_token must be '<OOV>', got {getattr(tokenizer, 'oov_token', None)!r}.")

            wi = tokenizer.word_index
            for must_have in ['<OOV>', 'movie', 'masterpiece', 'cinematography', 'soundtrack']:
                if must_have not in wi:
                    raise AssertionError(f"word_index missing '{must_have}'. Did you call fit_on_texts on the training sentences?")
            if wi.get('<OOV>') != 1:
                raise AssertionError(f"<OOV> token should be index 1, got {wi.get('<OOV>')}.")

            # ---- 2. Padded data shape (n_train, 10) ----
            arr = np.asarray(padded_data)
            if arr.ndim != 2:
                raise AssertionError(f"padded_data must be 2D, got shape {arr.shape}.")
            if arr.shape[1] != 10:
                raise AssertionError(f"padded_data sequences must have length 10 (maxlen), got {arr.shape[1]}.")
            if arr.shape[0] != len(train_sentences):
                raise AssertionError(f"padded_data should have one row per training sentence ({len(train_sentences)}), got {arr.shape[0]}.")

            # ---- 3. Padding direction = 'post' ----
            short_idx = min(range(len(train_sentences)), key=lambda i: len(train_sentences[i].split()))
            short_row = list(arr[short_idx])
            if short_row[-1] != 0:
                raise AssertionError("padded_data should have trailing zeros (padding='post'). Found non-zero at end of the shortest row.")
            if short_row[0] == 0:
                raise AssertionError("padded_data appears to use padding='pre' (leading zeros). Use padding='post'.")

            # ---- 4. Truncation: at least one row must be FULLY filled (no zeros) ----
            full_rows = [i for i in range(arr.shape[0]) if 0 not in arr[i]]
            if not full_rows:
                raise AssertionError("Truncation check failed: NO row is fully filled. At least one training sentence is longer than 10 tokens — its row must have zero zeros.")

            # ---- 5. Truncation direction = 'post' (keep FIRST 10 tokens, not last) ----
            raw_seqs = tokenizer.texts_to_sequences(train_sentences)
            for i in full_rows:
                if len(raw_seqs[i]) > 10:
                    expected_post = raw_seqs[i][:10]   # truncating='post' keeps the BEGINNING
                    expected_pre  = raw_seqs[i][-10:]  # truncating='pre'  keeps the END
                    if list(arr[i]) == expected_pre and list(arr[i]) != expected_post:
                        raise AssertionError("Truncation direction is wrong. Use truncating='post' (keep first 10 tokens, drop the tail).")
                    if list(arr[i]) != expected_post:
                        raise AssertionError(f"Row {i}: padded data does not match texts_to_sequences output. Did you re-fit the tokenizer or change something?")
                    break

            # ---- 6. OOV behaviour ----
            seq = oov_test_seq
            if seq and isinstance(seq[0], list):
                seq = seq[0]
            seq = list(seq)
            if 1 not in seq:
                raise AssertionError("oov_test_seq has no <OOV> (=1) token. Did you encode a sentence containing UNKNOWN words?")

            # ---- 7. decode_sequence function ----
            if not callable(decode_fn):
                raise AssertionError("decode_sequence must be a function (def decode_sequence(sequence, tokenizer): ...).")

            # 7a. Decode a normal padded row — must be a string with no '0' tokens left in
            decoded_first = decode_fn(arr[short_idx], tokenizer)
            if not isinstance(decoded_first, str):
                raise AssertionError(f"decode_sequence must return a string, got {type(decoded_first).__name__}.")
            if '0' in decoded_first.split():
                raise AssertionError("decode_sequence output contains '0' as a token. You must SKIP padding zeros, not convert them to '0'.")
            short_words = train_sentences[short_idx].lower().split()
            in_vocab_short = [w for w in short_words if w in wi]
            if in_vocab_short and not any(w in decoded_first.lower() for w in in_vocab_short):
                raise AssertionError(f"decode_sequence on the shortest row ({decoded_first!r}) does not contain any expected word from {in_vocab_short}.")

            # 7b. Decode the OOV test sequence — must contain '<OOV>' for unknown words
            decoded_oov = decode_fn(seq, tokenizer)
            if '<OOV>' not in decoded_oov:
                raise AssertionError(f"decode_sequence on the OOV test sequence should contain '<OOV>'. Got: {decoded_oov!r}.")

            # 7c. Decode round-trip — encode "movie" then decode, should contain "movie"
            sample_seq = tokenizer.texts_to_sequences(["movie"])[0]
            sample_padded = pad_sequences([sample_seq], maxlen=10, padding='post')[0]
            decoded_round = decode_fn(sample_padded, tokenizer)
            if 'movie' not in decoded_round:
                raise AssertionError(f"decode_sequence round-trip failed: encoding 'movie' then decoding gave {decoded_round!r}.")

            self.scores['Q7'] = self.max_scores['Q7']
            print("✅ Q7 Passed: Tokenizer + post-padding + post-truncation + decode_sequence all correct!")
        except AssertionError as e:
            self.scores['Q7'] = 0
            print(f"❌ Q7 Failed: {e}")
        except Exception as e:
            self.scores['Q7'] = 0
            print(f"❌ Q7 Failed with error: {e}")
        self.display_score()


grader = NLPGrader()
print("Setup complete. NLTK + Keras ready. Lights, camera, action! 🎬")


---
## 🧼 Q1 — Noise Removal & Normalization — 10 pts

Computers are extremely literal. To a computer, `"Amazing"`, `"amazing"`, and `"amazing!"` are three completely different tokens. Your first job is to **standardize** raw review text by:
1. Lowercasing every character.
2. Removing every punctuation mark (use `string.punctuation`).

> 💡 Tip from the worksheet: `str.maketrans('', '', string.punctuation)` builds a translation table that maps every punctuation char to *Nothing*.


In [ ]:
# =========================================================================
# Q1: WRITE A clean_text(text) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  clean_text(text)  that:
#   1. Converts the input string to LOWERCASE.
#   2. Removes ALL punctuation using string.punctuation.
#   3. Returns the cleaned string.
#
# Examples:
#   clean_text("AMAZING film!!! Loved it...")
#       -> 'amazing film loved it'
#   clean_text("OMG!!! The performances were ABSOLUTELY breathtaking #masterpiece")
#       -> 'omg the performances were absolutely breathtaking masterpiece'
#
# Notes:
#   - The grader normalizes whitespace, so multiple spaces left over from
#     removing punctuation will not be penalized.
#   - You MUST use string.punctuation (not a hand-typed list).
# =========================================================================

import string

# YOUR CODE HERE
def clean_text(text):
    # Step 1: lowercase
    # Step 2: remove punctuation
    # Step 3: return
    pass


# Quick sanity check
raw = "OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director"
print("Original:", raw)
print("Cleaned :", clean_text(raw))


In [ ]:
# Run this cell AFTER completing Q1 to check your answer
grader.check_q1(clean_text)


---
## ✂️ Q2 — Tokenization — 10 pts

Models don't read sentences; they read **tokens** (words). We need to chop the cleaned stream of text into a list of individual units.

We'll use the simple approach from the worksheet: **split on whitespace**.

> 💡 Be careful: `"  multiple   spaces  "` should still produce `['multiple', 'spaces']` — no empty strings, no leading/trailing blanks. (Hint: plain `.split()` with NO argument handles this for you.)


In [ ]:
# =========================================================================
# Q2: WRITE A tokenize(text) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  tokenize(text)  that:
#   - Takes a (presumably already-cleaned) string
#   - Returns a list of word tokens, splitting on whitespace.
#   - Must NOT contain any empty strings, even if input has multiple
#     consecutive spaces / leading / trailing whitespace.
#
# Examples:
#   tokenize("amazing film")          -> ['amazing', 'film']
#   tokenize("  extra   spaces  ")    -> ['extra', 'spaces']
# =========================================================================

# YOUR CODE HERE
def tokenize(text):
    pass


# Quick sanity check using your Q1 function
cleaned = clean_text("OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director")
tokens  = tokenize(cleaned)
print("Tokens     :", tokens)
print("Token count:", len(tokens))


In [ ]:
# Run this cell AFTER completing Q2 to check your answer
grader.check_q2(tokenize)


---
## 🛑 Q3 — Stop Word Removal — 10 pts

Words split into two camps:
- **Content words** carry meaning: *masterpiece, breathtaking, boring.*
- **Stop words** are grammatical glue with little standalone meaning: *the, is, was, at.*

For simple/light models like Naive Bayes we usually **drop** stop words to save space.

Use NLTK's English stop-word list. The setup cell already downloaded it for you.

> ⚠️ **Food for thought:** *"The movie was NOT good."* — naïvely removing stop words turns this into `['movie', 'good']` and the meaning **flips**. We accept this trade-off here for simple models — but remember it for real-world systems.


In [ ]:
# =========================================================================
# Q3: WRITE A remove_stopwords(tokens) FUNCTION
# =========================================================================
#
# Task: Define a function named EXACTLY  remove_stopwords(tokens)  that:
#   - Takes a list of lower-case tokens.
#   - Returns a new list with all NLTK English stop words removed.
#   - PRESERVES the original order of the remaining tokens.
#
# Required imports / setup:
#   from nltk.corpus import stopwords
#   stop_words = set(stopwords.words('english'))
#
# Example:
#   remove_stopwords(['the','movie','was','absolutely','amazing'])
#       -> ['movie', 'absolutely', 'amazing']
# =========================================================================

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

# YOUR CODE HERE
def remove_stopwords(tokens):
    pass


# Quick sanity check chained from Q1 + Q2
demo_tokens = tokenize(clean_text("OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director"))
print("Before:", demo_tokens)
print("After :", remove_stopwords(demo_tokens))


In [ ]:
# Run this cell AFTER completing Q3 to check your answer
grader.check_q3(remove_stopwords)


---
## 🌱 Q4 — Stemming vs. Lemmatization — 10 pts

If a model is trained on `"watching"` but a user types `"watch"`, the model won't realise they're related. We cut every word down to its **root form**.

| Approach | What it does | Speed | Accuracy |
|---|---|---|---|
| **Stemming** | Crude chopper — slices suffixes; sometimes makes non-words (`movies → movi`) | ⚡ fast | meh |
| **Lemmatization** | Dictionary lookup — returns real words (`movies → movie`) | 🐢 slower | ✅ accurate |

Build a dictionary that maps each movie-themed word to **both** its stem and its lemma.


In [ ]:
# =========================================================================
# Q4: BUILD root_forms — A DICT OF (stem, lemma) FOR EACH WORD
# =========================================================================
#
# Task:
#   For every word in the list `words` below, compute BOTH:
#       - its Porter stem        (PorterStemmer().stem(w))
#       - its WordNet lemma      (WordNetLemmatizer().lemmatize(w))   # default POS = 'n'
#   and store the result in a dictionary named EXACTLY:  root_forms
#
#   Format:
#       root_forms = {
#           'word_1': ('stem_1', 'lemma_1'),
#           'word_2': ('stem_2', 'lemma_2'),
#           ...
#       }
#
# Required (movie-themed) words:
#   performances, actors, watching, directed, movies, stories, running
# =========================================================================

from nltk.stem import PorterStemmer, WordNetLemmatizer

stemmer    = PorterStemmer()
lemmatizer = WordNetLemmatizer()

words = ["performances", "actors", "watching", "directed", "movies", "stories", "running"]

# YOUR CODE HERE
# root_forms = { ... }


# Pretty print
print(f"{'WORD':<14} {'STEM':<12} {'LEMMA':<12}")
print("-" * 38)
for w in words:
    s, l = root_forms[w]
    print(f"{w:<14} {s:<12} {l:<12}")


In [ ]:
# Run this cell AFTER completing Q4 to check your answer
grader.check_q4(root_forms)


---
## 📦 Q5 — Vocabulary Builder + Corpus Vectorizer — 15 pts ⭐⭐⭐

Up to now we've handled one sentence at a time. Real datasets contain **thousands of reviews** — and every Bag-of-Words vector across the dataset must use the **same** column ordering. That means we first build a global vocabulary, then vectorize every document against it.

You'll write **two cooperating functions**:

1. `build_vocabulary(corpus)` — given a *corpus* (list of token-lists), return a **sorted, alphabetical list of unique tokens**.
2. `vectorize_corpus(corpus, vocabulary)` — given a corpus and the global vocabulary, return a **2-D count matrix** (one row per document, one column per vocab word).

> ⚠️ **Trap-spotting checklist:**
> - The vocabulary MUST be **sorted alphabetically** (deterministic column ordering).
> - Bag-of-Words = **counts**, not just 0/1.
> - Empty documents must produce a zero row.
> - Tokens not in the vocabulary must be **silently ignored** (no error, no extra column).


In [ ]:
# =========================================================================
# Q5: BUILD VOCABULARY + VECTORIZE A CORPUS
# =========================================================================
#
# Two functions to write:
#
#   def build_vocabulary(corpus):
#       # corpus is a list of token-lists, e.g. [['great','movie'], ['boring','plot']]
#       # Return: a SORTED list of unique tokens (alphabetical order).
#
#   def vectorize_corpus(corpus, vocabulary):
#       # Return: a 2-D list (or numpy array) of shape (len(corpus), len(vocabulary))
#       #         where matrix[i][j] = count of vocabulary[j] in corpus[i].
#       # Tokens not in vocabulary are silently IGNORED.
#       # Empty documents -> a row of zeros.
#
# Worked example:
#   corpus = [
#       ['great', 'movie', 'great', 'acting'],
#       ['boring', 'movie', 'bad', 'acting'],
#       ['great', 'acting', 'amazing', 'film'],
#   ]
#   build_vocabulary(corpus)
#       -> ['acting', 'amazing', 'bad', 'boring', 'film', 'great', 'movie']
#
#   vectorize_corpus(corpus, vocab)
#       -> [[1, 0, 0, 0, 0, 2, 1],
#           [1, 0, 1, 1, 0, 0, 1],
#           [1, 1, 0, 0, 1, 1, 0]]
# =========================================================================

import numpy as np

# YOUR CODE HERE
def build_vocabulary(corpus):
    pass


def vectorize_corpus(corpus, vocabulary):
    pass


# Sanity check
demo_corpus = [
    ['great', 'movie', 'great', 'acting'],
    ['boring', 'movie', 'bad', 'acting'],
    ['great', 'acting', 'amazing', 'film'],
]
demo_vocab  = build_vocabulary(demo_corpus)
demo_matrix = vectorize_corpus(demo_corpus, demo_vocab)
print("Vocabulary :", demo_vocab)
print("BoW matrix :\n", np.asarray(demo_matrix))


In [ ]:
# Run this cell AFTER completing Q5 to check your answer
grader.check_q5(build_vocabulary, vectorize_corpus)


---
## 🔗 Q6 — Full Single-Text Pipeline — 15 pts ⭐⭐⭐

Time to wire all the single-text steps together. Build **one** function that takes a raw review and returns the final clean list of tokens, ready for vectorization.

**Required pipeline order — exact same order as the worksheet:**
1. Lowercase
2. Strip punctuation (`string.punctuation`)
3. Tokenize on whitespace
4. Drop NLTK English stop words
5. Lemmatize each remaining token (default `WordNetLemmatizer().lemmatize(w)` — POS defaults to noun)

> 💡 You may call your earlier functions (`clean_text`, `tokenize`, `remove_stopwords`) — that's the whole point of writing them as functions! 🎯


In [ ]:
# =========================================================================
# Q6: WRITE THE COMPLETE preprocess(text) PIPELINE
# =========================================================================
#
# Task: Define a function named EXACTLY  preprocess(text)  that:
#   - Performs ALL FIVE STEPS in this order:
#       (1) lowercase
#       (2) remove punctuation                  (string.punctuation)
#       (3) split on whitespace                 (no empty strings)
#       (4) drop NLTK English stop words
#       (5) lemmatize each remaining token      (WordNetLemmatizer, default POS)
#   - Returns a list of clean lemma tokens.
#
# Example:
#   preprocess("OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director")
#       -> ['omg', 'performance', 'absolutely', 'breathtaking', 'literally', 'cried', 'watching', 'masterpiece', 'director']
#
# Note: 'performances' becomes 'performance' (lemmatizer found the noun lemma).
#       'watching' / 'cried' stay as-is because default POS='n'.
# =========================================================================

import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words_set = set(stopwords.words('english'))
lemmatizer_q6  = WordNetLemmatizer()

# YOUR CODE HERE
def preprocess(text):
    pass


# Sanity check
raw = "OMG!!! The performances were ABSOLUTELY breathtaking... I literally cried watching it :) #masterpiece @director"
print("Raw  :", raw)
print("Final:", preprocess(raw))


In [ ]:
# Run this cell AFTER completing Q6 to check your answer
grader.check_q6(preprocess)


---
## 🚀 Q7 — Keras Tokenizer + Padding + Truncation + Decoder — 30 pts ⭐⭐⭐⭐⭐ (The Final Boss)

Real-world neural networks expect **integer sequences of equal length** — not lists of strings of varying length. Keras gives us two helpers, plus you'll write a **third helper of your own**: a `decode_sequence` function that turns a padded integer row back into human-readable text.

There's an additional twist this time: the training reviews vary a lot in length. Some are **shorter than `maxlen`** (so they need padding), others are **longer than `maxlen`** (so they need *truncating*). Keras supports both directions for both operations:

| Direction | `padding` (when too short) | `truncating` (when too long) |
|---|---|---|
| `'pre'`  | zeros at the **start** | drop tokens from the **start** |
| `'post'` | zeros at the **end** | drop tokens from the **end** |

We will use `'post'` for **both**.

Both helpers are imported in the setup cell:
```python
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
```

Your job: take 8 movie reviews of varying length, fit a tokenizer, convert them to integer sequences, **pad and truncate** them all to length **10**, encode an OOV-containing test review, AND write a `decode_sequence(sequence, tokenizer)` function that converts a padded integer row back into a string.


In [ ]:
# =========================================================================
# Q7: KERAS TOKENIZER + PADDING + TRUNCATION + DECODER (The Final Boss!)
# =========================================================================
#
# Training reviews — 8 sentences. NOTE varying length:
#   reviews 0,1,2,3,7  : short  (need PADDING)
#   review  5          : LONG (>10 tokens) — needs TRUNCATION
#   review  6          : long-ish — possibly needs truncation too
train_sentences = [
    "absolutely loved this movie",                                               #  4 tokens
    "fantastic performance by the lead actor",                                   #  6 tokens
    "the cinematography was breathtaking",                                       #  4 tokens
    "boring predictable plot disappointing ending",                              #  5 tokens
    "i hated every single minute of this terrible film",                         #  9 tokens
    "best movie of the year hands down a true masterpiece for cinema lovers",    # 13 tokens — TRUNCATE!
    "weak story but the soundtrack was great and the acting was passable",      # 11 tokens — TRUNCATE!
    "loved it",                                                                  #  2 tokens
]
#
# Test review (contains OOV words 'pizza' and 'urgent' that aren't in training):
test_sentence = "urgent pizza scene was masterpiece"
#
# ----------------------- TASKS -----------------------
#  1. Create a Keras Tokenizer named EXACTLY  tokenizer  with:
#         num_words = 100
#         oov_token = '<OOV>'
#
#  2. FIT the tokenizer on train_sentences (fit_on_texts).
#
#  3. Convert train_sentences into integer sequences using
#     texts_to_sequences. Store the result in a variable named  sequences.
#
#  4. Pad/truncate sequences with:
#         maxlen     = 10
#         padding    = 'post'      (zeros at the END for short sentences)
#         truncating = 'post'      (drop tail tokens for long sentences)
#     Store the result in a variable named EXACTLY  padded_data.
#
#  5. Encode the test_sentence using texts_to_sequences and store the
#     resulting sequence (still nested list-of-lists is fine) in a variable
#     named EXACTLY  oov_test_seq.
#
#  6. WRITE A FUNCTION  decode_sequence(sequence, tokenizer)  that:
#       - Takes a 1-D sequence of ints (e.g. one row of padded_data)
#       - Uses tokenizer.index_word to map each non-zero int back to its word
#       - SKIPS zeros (they are padding, not real tokens!)
#       - Joins the resulting words with single spaces and returns the string
#       - Unknown int (=1) maps to '<OOV>' just like any other word
#
#     Hint:
#         words = [tokenizer.index_word[i] for i in sequence if i != 0]
#         return ' '.join(words)
#
# Expected behaviour:
#   - padded_data.shape == (8, 10)
#   - padded_data[7]    starts with the words of "loved it" then trailing zeros
#   - padded_data[5]    is FULLY filled (no zeros) — it was truncated to first 10
#   - oov_test_seq      contains the integer 1 (<OOV>) somewhere
#   - decode_sequence(padded_data[0], tokenizer) returns a clean string with NO '0' tokens
# =========================================================================

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --- Step 1: Create the tokenizer ---
# YOUR CODE HERE
# tokenizer = ...


# --- Step 2: Fit on training sentences ---
# YOUR CODE HERE


# --- Step 3: Convert to integer sequences ---
# YOUR CODE HERE
# sequences = ...


# --- Step 4: Pad/truncate to length 10, both 'post' ---
# YOUR CODE HERE
# padded_data = ...


# --- Step 5: Encode the OOV test review ---
# YOUR CODE HERE
# oov_test_seq = ...


# --- Step 6: Write decode_sequence(sequence, tokenizer) ---
# YOUR CODE HERE
def decode_sequence(sequence, tokenizer):
    pass


# Inspect your work
print("Word index (first 12):")
for w, i in list(tokenizer.word_index.items())[:12]:
    print(f"  {w!r:>20} -> {i}")
print()
print("padded_data shape :", padded_data.shape)
print("Short row (idx 7) :", padded_data[7])
print("Long row  (idx 5) :", padded_data[5], '   <-- should be FULLY filled (no zeros)')
print("OOV test sentence :", test_sentence)
print("OOV test sequence :", oov_test_seq)
print()
print("--- Decoder demo ---")
print("Decoded row 0     :", decode_sequence(padded_data[0], tokenizer))
print("Decoded row 7     :", decode_sequence(padded_data[7], tokenizer))
print("Decoded OOV test  :", decode_sequence(oov_test_seq[0] if isinstance(oov_test_seq[0], list) else oov_test_seq, tokenizer))


In [ ]:
# Run this cell AFTER completing Q7 to check your answer
grader.check_q7(tokenizer, padded_data, oov_test_seq, decode_sequence, train_sentences)


---
## ✅ Submission

When all seven checks show green:
1. Confirm the **NLP Live Score** at the top reads **100 / 100**.
2. From the Colab menu choose **File → Download → Download .ipynb**.
3. Send the downloaded notebook back to your instructor.

Make sure all cells have been **run** so the outputs (and your score) are visible in the saved file. 🎉

### 🎯 What you just built
You took raw, messy movie reviews from chaos all the way down to a clean **(8, 10) integer tensor** — *and* you can decode it back into human-readable text. Every modern NLP system — including ChatGPT — starts with exactly this kind of pipeline.

Next up (next week): plug your `padded_data` into an `Embedding` layer, train an RNN, and you've got a real sentiment classifier. 🚀🎬

# **RNN Model**

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Embedding

# 1. Setup Data
train_sentences = [
    'The movie was truly amazing and exciting',   # Positive (1)
    'I absolutely loved the delicious food',      # Positive (1)
    'This is the best day ever',                 # Positive (1)
    'The service was terrible and very slow',    # Negative (0)
    'I really hated the boring movie',           # Negative (0)
    'This is the worst experience ever'          # Negative (0)
]
train_labels = np.array([1,1,1,0,0,0])

# 2. Pipeline
# A. Tokenize & Integer Encode
tokenizer = Tokenizer(num_words=100, oov_token="<OOV>")
tokenizer.fit_on_texts(train_sentences)
sequences = tokenizer.texts_to_sequences(train_sentences)

# B. Padding
max_length = 8
padded_data = pad_sequences(sequences, maxlen=max_length, padding="post")

print("Word Index:", tokenizer.word_index)
print("Padded Data:\n", padded_data)

# 3. The model
model = Sequential()

# Layer 1: Embeding(The DNA Learner)
model.add(Embedding(input_dim=100, output_dim=8))

# Layer 2: SimpleRNN (The Sequence Reader)
model.add(SimpleRNN(16))

# Layer 3: Output (Decision)
model.add(Dense(1, activation="sigmoid"))

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# 4. Train
model.fit(padded_data, train_labels, epochs=50)

# 5. Predict
test_text = ["I loved the exciting movie", "The food was terrible"]
test_seq = tokenizer.texts_to_sequences(test_text)
test_pad = pad_sequences(test_seq, maxlen=max_length, padding="post")

print("\n--- Test Results ---")
preds = model.predict(test_pad)
print("Predictions:", preds)

Word Index: {'<OOV>': 1, 'the': 2, 'movie': 3, 'was': 4, 'and': 5, 'i': 6, 'this': 7, 'is': 8, 'ever': 9, 'truly': 10, 'amazing': 11, 'exciting': 12, 'absolutely': 13, 'loved': 14, 'delicious': 15, 'food': 16, 'best': 17, 'day': 18, 'service': 19, 'terrible': 20, 'very': 21, 'slow': 22, 'really': 23, 'hated': 24, 'boring': 25, 'worst': 26, 'experience': 27}
Padded Data:
 [[ 2  3  4 10 11  5 12  0]
 [ 6 13 14  2 15 16  0  0]
 [ 7  8  2 17 18  9  0  0]
 [ 2 19  4 20  5 21 22  0]
 [ 6 23 24  2 25  3  0  0]
 [ 7  8  2 26 27  9  0  0]]
Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - accuracy: 0.5000 - loss: 0.7054
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 170ms/step - accuracy: 0.6667 - loss: 0.7002
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 0.6667 - loss: 0.6952
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.6667 - loss: 0.6902
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 0.8333 - loss: 0.6853
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/s